# Task 3 — White-box Testing (Branch Coverage)

Branch-coverage tests for two `TrafficLight` methods (both CC 7, different loop types):
- `execute_cycle` — for-loop
- `apply_manual_override` — while-loop

We chose branch coverage over condition coverage because neither method contains compound `and`/`or` decisions — condition coverage would produce the same tests. Branch also subsumes statement and method coverage, which we verify with `coverage.py`.

## 1. Setup
Import the class from this task's own `src/` copy.

In [8]:
import os, sys, time, subprocess, textwrap, inspect
import requests

SRC_DIR  = os.path.abspath("src")     # this task's own copy of the code under test
TEST_DIR = os.path.abspath("tests")
sys.path.insert(0, SRC_DIR)
from src.traffic_light import TrafficLight

## 2. Methods under test
Print the source so the branch analysis below is tied to the actual code.

To note: `is_safe_state()` flags any state where both directions share a colour as *unsafe* — including both RED. So a "safe" state always needs two *different* colours.

In [9]:
for name in ("execute_cycle", "apply_manual_override"):
    print("="*68, name, "="*(66-len(name)), sep="\n")
    print(inspect.getsource(getattr(TrafficLight, name)))

execute_cycle
    def execute_cycle(self):
        if self.manual_mode:
            return self.apply_manual_override([])
        if not self.is_safe_state():
            self.set_all_red()
            return
        for direction in self.directions:
            if self.current_directions[direction] == "GREEN":
                if self.check_synchronization(direction):
                    self.yellow_transition(direction)
                else:
                    self.set_all_red()
                    return
            elif self.current_directions[direction] == "YELLOW":
                self.red_clearance_phase(direction)

apply_manual_override
    def apply_manual_override(self, commands):
        """Refactored: consumes actions from `commands` (a list) instead of input().
        Returns True if a safe state is reached, False on QUIT or running out of
        commands. Raises ValueError on a MANUAL_OVERRIDE action.
        Control flow (while + nested for + branches) is unchanged, so

## 3. Branch analysis

We identified every decision in each method and designed the minimal inputs to force each one both True and False.

### `execute_cycle` (4 tests)

| Test | State | Branches forced |
|---|---|---|
| T1 | `manual_mode=True` | D1=T |
| T2 | auto, both RED (unsafe) | D1=F, D2=T |
| T3 | NS=GREEN, EW=RED | D2=F, D4=T/F, D5=T, D6=F |
| T4 | NS=YELLOW, EW=RED | D6=T |

D5=False is **infeasible**: it requires two greens, but that's unsafe and already caught at D2 before the loop. Dead code — documented, not tested.

### `apply_manual_override` (6 tests)

| Test | Setup | Branches forced |
|---|---|---|
| M1 | NS=GREEN, EW=RED, `[]` | W=F (skip loop) |
| M2 | both RED, emergency=`"NS"` | W=T, E=T |
| M3 | both RED, `[]` | E=F, C1=T |
| M4 | both RED, `["MANUAL_OVERRIDE"]` | C2=T |
| M5 | both RED, `["QUIT"]` | C3=T |
| M6 | both RED, `["GREEN","RED"]` | C3=F, loop back-edge, W=F exit |

All 6 branches feasible. M6 is the only test that makes the while-loop iterate and then stop.

## 4. Generate pytest code with Ollama
We fed the branch tables to llama3 to draft the pytest file. Raw output saved as `tests/_generated_draft_whitebox.py` (generation evidence).

In [10]:
OLLAMA_URL, MODEL = "http://localhost:11434/api/generate", "llama3"

prompt = textwrap.dedent("""\
Write a pytest file (Python ONLY, no prose, no markdown fences) for BRANCH-COVERAGE
testing of TrafficLight (import after adding ../src to sys.path). Helper make_light()
returns a TrafficLight with colors {"GREEN":0,"YELLOW":0,"RED":0}. Quirk: a SAFE state
needs the two directions to DIFFER.
execute_cycle: T1 manual_mode=True->False; T2 both RED->None,stays RED; T3 NS=GREEN,EW=RED
->NS becomes RED; T4 NS=YELLOW,EW=RED->NS stays YELLOW.
apply_manual_override(commands): M1 NS=GREEN,EW=RED,[]->True; M2 both RED,emergency_request='NS',[]
->True and NS GREEN; M3 both RED,[]->False; M4 both RED,['MANUAL_OVERRIDE']->ValueError;
M5 both RED,['QUIT']->False; M6 both RED,['GREEN','RED']->True and NS=GREEN,EW=RED.
Output only the file content.
""")

draft = os.path.join(TEST_DIR, "_generated_draft_whitebox.py")
t0 = time.time()
r = requests.post(OLLAMA_URL, json={"model": MODEL, "prompt": prompt, "stream": False}, timeout=600)
elapsed = time.time() - t0
text = r.json().get("response", "")
if "```" in text:
    text = max(text.split("```"), key=len).replace("python", "", 1).strip()
open(draft, "w", encoding="utf-8").write(text)
print(f"generated {len(text)} chars in {elapsed:.1f}s -> {draft}")

generated 3356 chars in 32.4s -> c:\Users\hiromi\Documents\GitHub\traffic-light-testing\task3\tests\_generated_draft_whitebox.py


## 5. Verified test suite
Hand-reviewed and finalized version of the Ollama draft. `make_light()` zeros all durations so `time.sleep` is instant.

In [11]:
print(open(os.path.join(TEST_DIR, "test_traffic_light_whitebox.py"), encoding="utf-8").read())

"""
Task 3 - White-box (BRANCH COVERAGE) tests for the Task 2 TrafficLight code.

Target methods (reused from Task 2, both CC = 7, different loop types):
  * execute_cycle          -> for-loop
  * apply_manual_override  -> while-loop

Each test is labelled with the decision/branch it forces. Durations are zeroed
(make_light) so time.sleep() is instant -> the suite is fast and deterministic.

execute_cycle branch map:
  T1 D1=True | T2 D1=False,D2=True | T3 D2=False,D4=T/F,D5=True,D6=False
  T4 D6=True | D5=False is INFEASIBLE (two greens => unsafe, caught at D2)
apply_manual_override branch map:
  M1 W=False | M2 W=T,E=True | M3 E=False,C1=True | M4 C2=True
  M5 C3=True | M6 C3=False + loop back-edge + W=False exit
"""
import os, sys
import pytest

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from src.traffic_light import TrafficLight


def make_light():
    """A TrafficLight with all durations zeroed so time.sleep is instant."""
    tl = TrafficLight()
    

## 6. Run the tests

Execute the full suite with `pytest -v`. We expect **10 passed** — 4 for `execute_cycle` and 6 for `apply_manual_override`.

In [12]:
print(subprocess.run([sys.executable, "-m", "pytest", "tests/", "-v"],
                     capture_output=True, text=True).stdout[-2000:])

============================= test session starts =============================
platform win32 -- Python 3.12.1, pytest-9.0.3, pluggy-1.6.0 -- c:\Users\hiromi\Documents\GitHub\traffic-light-testing\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\hiromi\Documents\GitHub\traffic-light-testing\task3
plugins: anyio-4.13.0
collecting ... collected 10 items

tests/test_traffic_light_whitebox.py::test_ec_T1_manual_mode_true PASSED [ 10%]
tests/test_traffic_light_whitebox.py::test_ec_T2_auto_unsafe PASSED      [ 20%]
tests/test_traffic_light_whitebox.py::test_ec_T3_green_synchronized PASSED [ 30%]
tests/test_traffic_light_whitebox.py::test_ec_T4_yellow_branch PASSED    [ 40%]
tests/test_traffic_light_whitebox.py::test_amo_M1_already_safe PASSED    [ 50%]
tests/test_traffic_light_whitebox.py::test_amo_M2_emergency PASSED       [ 60%]
tests/test_traffic_light_whitebox.py::test_amo_M3_run_out_of_commands PASSED [ 70%]
tests/test_traffic_light_whitebox.py::test_amo_M4_manual_ove

## 7. Coverage measurement
`coverage.py --branch` gives objective numbers. The whole-file ~75% is because untargeted methods aren't covered — expected and out of scope.

In [13]:
subprocess.run([sys.executable, "-m", "coverage", "run", "--branch", "-m", "pytest", "tests/", "-q"],
               capture_output=True, text=True)
print(subprocess.run([sys.executable, "-m", "coverage", "report", "-m", "--include=*traffic_light.py"],
                     capture_output=True, text=True).stdout)

Name                   Stmts   Miss Branch BrPart  Cover   Missing
------------------------------------------------------------------
src\traffic_light.py      94     22     52      5    75%   22-23, 32, 36-37, 42, 47, 53-57, 60-62, 65-69, 72, 119
------------------------------------------------------------------
TOTAL                     94     22     52      5    75%



## 8. Results

- **`apply_manual_override`**: 100% statement + branch coverage.
- **`execute_cycle`**: all feasible branches covered. The only miss is D5=False (lines 22–23) — the infeasible branch confirmed by `coverage.py`.
- Branch coverage subsumes statement and method coverage → three ladder rungs from one test set.
- The infeasible branch is the white-box twin of the infeasible partition removed in Task 2 — same dead code, found from the inside this time.